# Chromatic focusing

Compare a cascade optimized to put a whole spectrum into **one** focal plane with a cascade optimized so that **each sampled energy focuses at its own plane**. The axial \(I(z, E)\) map is the main diagnostic: a vertical stripe is achromatic (all colors at one \(z\)), a diagonal is controlled chromatic walk, and off-diagonal brightness is leakage of the wrong color into the wrong plane.

Run the experiment first:

```bash
python paper/experiments/chromatic_focusing.py --save-dir paper_data
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path


In [ ]:
import sys

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

from src.util import colors_list


In [ ]:
matplotlib.rcParams["figure.dpi"] = 200
matplotlib.rcParams.update({"font.size": 18})


In [ ]:
PREFIX = "chromatic_focusing"


def _find_latest_results(prefix: str) -> Path:
    search_dirs = [repo_root / "paper_data", repo_root / "outputs"]
    candidates = []
    for directory in search_dirs:
        if directory.exists():
            candidates.extend(directory.glob(f"{prefix}_results*.npz"))
    if not candidates:
        raise FileNotFoundError(
            f"No {prefix}_results*.npz files found in paper_data/ or outputs/. "
            "Run paper/experiments/chromatic_focusing.py first."
        )
    exact = [p for p in candidates if p.name == f"{prefix}_results.npz"]
    if exact:
        return exact[0]
    return max(candidates, key=lambda p: p.stat().st_mtime)


results_path = _find_latest_results(PREFIX)
params_path = Path(str(results_path).replace("_results", "_params").replace(".npz", ".npy"))
params = np.load(params_path, allow_pickle=True).item()
results = np.load(results_path, allow_pickle=True)
print(results_path)


In [ ]:
f = float(params["f"])
energies_kev = np.asarray(results["energies_ev"]) / 1e3
z_eval = np.asarray(results["z_eval"])
z_last = np.asarray(results["z_last_per_wavelength"])
z_focus_single = np.asarray(results["z_focus_single"])
z_focus_chromatic = np.asarray(results["z_focus_chromatic"])
z_focus_fzp = np.asarray(results["z_focus_fzp"])
fzp_theory_z = np.asarray(results["fzp_theory_z"])
inmask_single = np.asarray(results["inmask_single"])
inmask_chromatic = np.asarray(results["inmask_chromatic"])
inmask_fzp = np.asarray(results["inmask_fzp"])
crosstalk_single = np.asarray(results["crosstalk_single"])
crosstalk_chromatic = np.asarray(results["crosstalk_chromatic"])
crosstalk_fzp = np.asarray(results["crosstalk_fzp"])
profiles_single = np.asarray(results["profiles_single"])
profiles_chromatic = np.asarray(results["profiles_chromatic"])
x_crop = np.asarray(results["x_crop"])
n_wvl = energies_kev.size
lam_colors = colors_list(n_wvl)
energy_labels = [f"{e:.2f} keV" for e in energies_kev]


def _row_normalize(crosstalk):
    row_sum = crosstalk.sum(axis=1, keepdims=True)
    row_sum = np.where(row_sum > 0, row_sum, 1.0)
    return crosstalk / row_sum


## Axial chromatic maps

Each panel is in-mask power versus defocus and photon energy. White markers are the prescribed \(z(E)\). The dotted line is the FZP law \(f \propto E\). Read leakage as brightness away from the intended locus: a bright vertical stripe means every color is still focusing at one plane; a tight diagonal means the stack put each color where it was asked to.


In [ ]:
dz_um = (z_eval - f) * 1e6
z_last_um = (z_last - f) * 1e6
fzp_theory_um = (fzp_theory_z - f) * 1e6
maps = [inmask_single, inmask_chromatic, inmask_fzp]
titles = ["single-plane cascade", "chromatic cascade", "Fresnel zone plate"]
vmax = max(float(m.max()) for m in maps)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), sharey=True)
im = None
for ax, data, title in zip(axes, maps, titles):
    im = ax.pcolormesh(
        dz_um,
        energies_kev,
        data,
        shading="auto",
        cmap="inferno",
        vmin=0.0,
        vmax=vmax,
    )
    ax.plot(z_last_um, energies_kev, "o", color="white", ms=6, mew=0.8, mec="k", label="prescribed $z(E)$")
    ax.plot(fzp_theory_um, energies_kev, "c--", lw=1.5, label=r"FZP $f \propto E$")
    ax.axvline(0.0, color="w", ls=":", lw=1.0, alpha=0.7)
    ax.set_xlabel(r"$z - f$ ($\mu$m)")
    ax.set_title(title)
axes[0].set_ylabel("energy (keV)")
axes[2].legend(loc="upper right", fontsize=11, framealpha=0.85)
cbar = fig.colorbar(im, ax=axes, shrink=0.92, pad=0.02)
cbar.set_label("in-mask power")
fig.suptitle("Where each color focuses", y=1.03)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot((z_focus_single - f) * 1e6, energies_kev, "o-", color="0.3", label="single-plane cascade")
ax.plot((z_focus_chromatic - f) * 1e6, energies_kev, "s-", color="C3", label="chromatic cascade")
ax.plot((z_focus_fzp - f) * 1e6, energies_kev, "^--", color="C2", label="FZP (simulated)")
ax.plot(z_last_um, energies_kev, "k:", lw=2, label="prescribed $z(E)$")
ax.plot(fzp_theory_um, energies_kev, "c--", lw=1.5, label=r"FZP $f \propto E$")
ax.axvline(0.0, color="0.5", ls="--", lw=1)
ax.set_xlabel(r"$z_\mathrm{focus} - f$ ($\mu$m)")
ax.set_ylabel("energy (keV)")
ax.legend(fontsize=12, loc="best")
ax.set_title("Chromatic focal shift")
plt.show()


## Crosstalk at the prescribed planes

Rows are evaluation planes (labeled by the energy that was asked to focus there). Columns are the actual photon energy. A diagonal matrix means each plane is dominated by its target color; off-diagonal entries are the wrong wavelength leaking into that focal spot. Values are row-normalized, so each plane sums to one.


In [ ]:
crosstalk_maps = [
    _row_normalize(crosstalk_single),
    _row_normalize(crosstalk_chromatic),
    _row_normalize(crosstalk_fzp),
]
fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.0))
for ax, data, title in zip(axes, crosstalk_maps, titles):
    im = ax.imshow(data, origin="lower", vmin=0.0, vmax=1.0, cmap="magma")
    ax.set_xticks(np.arange(n_wvl))
    ax.set_yticks(np.arange(n_wvl))
    ax.set_xticklabels(energy_labels, rotation=45, ha="right", fontsize=11)
    ax.set_yticklabels(energy_labels, fontsize=11)
    ax.set_xlabel("photon energy")
    ax.set_title(title)
    for i in range(n_wvl):
        for j in range(n_wvl):
            ax.text(j, i, f"{data[i, j]:.2f}", ha="center", va="center", color="w" if data[i, j] < 0.6 else "k", fontsize=9)
axes[0].set_ylabel("evaluation plane (target energy)")
cbar = fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02)
cbar.set_label("fraction of in-mask power")
fig.suptitle("Who is in each focal spot?", y=1.03)
plt.show()


## Focal-spot lineouts, filtered by wavelength

Each row is one prescribed plane. Solid traces are the chromatic cascade; dashed traces are the single-plane cascade. The thick curve is the target energy for that plane; thin curves are the other wavelengths leaking into the same plane.


In [ ]:
x_nm = x_crop * 1e9
fig, axes = plt.subplots(n_wvl, 1, figsize=(8.5, 2.4 * n_wvl), sharex=True)
if n_wvl == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    for j in range(n_wvl):
        lw = 2.6 if j == i else 1.1
        alpha = 1.0 if j == i else 0.55
        ax.plot(
            x_nm,
            profiles_chromatic[i, j],
            color=lam_colors[j],
            lw=lw,
            alpha=alpha,
            label=energy_labels[j] if i == 0 else None,
        )
        ax.plot(
            x_nm,
            profiles_single[i, j],
            color=lam_colors[j],
            lw=1.2,
            ls="--",
            alpha=0.8 if j == i else 0.35,
        )
    ax.set_ylabel("intensity")
    ax.set_title(f"plane for {energy_labels[i]}  ($z-f$ = {z_last_um[i]:.0f} $\mu$m)", fontsize=15)
axes[0].plot([], [], color="k", lw=2.0, label="chromatic cascade")
axes[0].plot([], [], color="k", lw=1.2, ls="--", label="single-plane cascade")
axes[0].legend(fontsize=11, ncol=2, loc="upper right")
axes[-1].set_xlabel(r"$x$ (nm)")
fig.suptitle("Target color vs leakage at each plane", y=1.01)
plt.tight_layout()
plt.show()
